# Shailaputri v1 — anomalies across a server's 38 metrics (SMD)

Model 1 of the nine, second dataset. **Data:** SMD, the Server Machine Dataset (MIT, 28 real servers, pinned commit). **Method:** Shailaputri's causal per-metric features, aggregated across a machine's metrics each minute, into LightGBM; an onset-weighted variant to flag earlier. **Split:** leave one machine group out. **Baselines:** max robust z and per-machine PCA. Scored event by event, never point-adjusted.

Results: `docs/research/models/shailaputri-smd.md`.

In [ ]:
import os
import subprocess
import sys

# Run on Kaggle or Colab. Clones this repository and installs the two packages.
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core",
                "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from sati.detect.smd import REVISION
from sati.sources import smd

m = smd.machine("machine-1-1", Path("cache/smd"), REVISION)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(m["test"][:, :6])
for start, end in smd.ranges(m["labels"]):
    ax.axvspan(start, end, color="red", alpha=0.2)
ax.set_title("machine-1-1, six of 38 metrics, labelled anomalies in red")
plt.show()

In [ ]:
import json
import subprocess
import sys

import pandas as pd

subprocess.run([sys.executable, "-m", "sati.detect.smd", "--cache", "cache/smd", "--out", "results/shailaputri-smd"], check=True, cwd="PASHUPATASTRA/packages/sati")
r = json.load(open("PASHUPATASTRA/packages/sati/results/shailaputri-smd/results.json"))
pd.DataFrame(r["mean_across_groups"]).T